# Voyexa AI — Model 2: Popularity Rating Regression
Predicts `Popularity_Rating` from a place's attributes (cost, terrain, tags, difficulty).
Useful for: understanding what drives popularity, and scoring places consistently even when review volume is low.

In [1]:
import pandas as pd
import numpy as np
import joblib
import os
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

DATA_PATH = os.path.join("..", "..", "data", "processed", "places_processed.csv")
ARTIFACTS_DIR = os.path.join("..", "artifacts")
os.makedirs(ARTIFACTS_DIR, exist_ok=True)

df = pd.read_csv(DATA_PATH)
df.shape

(1163, 202)

## Build feature matrix
`Hidden_Gem_Score`/`Is_Hidden_Gem` and `Number_Of_Reviewpoints` are excluded — the goal is to predict popularity from place *attributes* alone, not from signals derived from popularity itself or review volume.

In [2]:
interest_cols = [c for c in df.columns if c.startswith("interest_")]
suitable_cols = [c for c in df.columns if c.startswith("suitable_")]

feature_cols = interest_cols + suitable_cols + [
    "difficulty_score", "activity_cost_avg", "hotel_cost_avg", "food_cost_avg",
    "total_daily_cost_avg", "Entry_Fee_INR",
]
zone_dummies = pd.get_dummies(df["Zone"], prefix="zone")
terrain_dummies = pd.get_dummies(df["Terrain_Type"], prefix="terrain")

X = pd.concat([df[feature_cols], zone_dummies, terrain_dummies], axis=1)
y = df["Popularity_Rating"]

print(f"Features: {X.shape[1]}, Rows: {X.shape[0]}")
y.describe()

Features: 185, Rows: 1163


count    1163.000000
mean        4.512124
std         0.182729
min         3.900000
25%         4.400000
50%         4.500000
75%         4.600000
max         5.000000
Name: Popularity_Rating, dtype: float64

## Train / test split + Random Forest regressor

In [3]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

reg = RandomForestRegressor(n_estimators=300, max_depth=12, random_state=42)
reg.fit(X_train, y_train)
y_pred = reg.predict(X_test)

## Evaluate

In [4]:
rmse = mean_squared_error(y_test, y_pred) ** 0.5
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"RMSE: {rmse:.3f}")
print(f"MAE:  {mae:.3f}")
print(f"R2:   {r2:.3f}")
print(f"Target range: {y.min():.1f} - {y.max():.1f}, mean {y.mean():.2f}")

RMSE: 0.133
MAE:  0.106
R2:   0.456
Target range: 3.9 - 5.0, mean 4.51


## Feature importance

In [5]:
importances = pd.Series(reg.feature_importances_, index=X.columns).sort_values(ascending=False)
importances.head(10)

total_daily_cost_avg    0.203106
food_cost_avg           0.159198
interest_spiritual      0.105529
activity_cost_avg       0.069199
hotel_cost_avg          0.048439
Entry_Fee_INR           0.041683
difficulty_score        0.019063
interest_cultural       0.018362
interest_religion       0.017567
interest_nature         0.016389
dtype: float64

## Save model artifacts

In [6]:
joblib.dump(reg, os.path.join(ARTIFACTS_DIR, "popularity_regressor.pkl"))
joblib.dump(list(X.columns), os.path.join(ARTIFACTS_DIR, "popularity_features.pkl"))
print("Saved model + feature list to ml/artifacts/")

Saved model + feature list to ml/artifacts/
